# Connection to JDBC SQL Server Injection

In [0]:
server = "asql-salesproject-yash-s1.database.windows.net"
port = "1433"
database = "ASQL_SalesProject_Yash_Ingestion"
username = "sales"
password = "Yash1234"
#JDBC URL
jdbc_url_src = f"jdbc:sqlserver://{server}:{port};databaseName={database}"
#Connection Properties
connection_properties_src = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Connection to JDBC SQL Server Integration

In [0]:
server = "asql-salesproject-yash-s1.database.windows.net"
port = "1433"
database = "ASQL_SalesProject_Yash_Integration"
username = "sales"
password = "Yash1234"
#JDBC URL
jdbc_url_tgt = f"jdbc:sqlserver://{server}:{port};databaseName={database}"
#Connection Properties
connection_properties_tgt = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

##For DIM_PRODUCT TABLE 

In [0]:
from pyspark.sql.functions import *

# =========================================================
# PRODUCT DIMENSION SCD1 LOAD
# =========================================================

# ---------------------------------------------------------
# 1. READ SOURCE TABLES
# ---------------------------------------------------------

# PRODUCT SOURCE TABLE
df_PRODUCT_src = spark.read.jdbc(url=jdbc_url_src,table="dbo.PRODUCT",properties=connection_properties_src)

# PRODUCT NAME LOOKUP TABLE
df_PRODUCT_NAME_LKP = spark.read.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_PRODUCT_NAME_LKP",properties=connection_properties_tgt)

# TARGET TABLE
df_PRODUCT_tgt = spark.read.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_PRODUCT",properties=connection_properties_tgt)

# ---------------------------------------------------------
# 2. JOIN PRODUCT WITH PRODUCT_NAME LOOKUP
# ---------------------------------------------------------

df_PRODUCT_final_src = (df_PRODUCT_src.alias("src").join(df_PRODUCT_NAME_LKP.alias("lkp"),col("src.PRODUCT_NUMBER") == col("lkp.PRODUCT_NUMBER"),"left")
                        .select(col("src.PRODUCT_NUMBER"),col("lkp.PRODUCT_NAME").alias("PRODUCT_NAME"),col("src.BASE_PRODUCT_NUMBER"),
                                col("src.PRODUCT_COLOR_CODE").alias("PRODUCT_COLOR"),col("src.PRODUCT_SIZE_CODE").alias("PRODUCT_SIZE"),
                                col("src.PRODUCT_BRAND_CODE").alias("PRODUCT_BRAND"),col("src.PRODUCTION_COST"),col("src.INTRODUCTION_DATE"),
                                col("src.DISCONTINUED_DATE"),col("src.SOURCE_ID"),col("src.DataDate").cast("date").alias("DATA_DATE")
                                )
                        )

# ---------------------------------------------------------
# 3. UPDATED RECORDS
# BUSINESS KEY = PRODUCT_NUMBER
# ---------------------------------------------------------

updated_PRODUCT_df = (df_PRODUCT_final_src.alias("src").join(df_PRODUCT_tgt.alias("tgt"),"PRODUCT_NUMBER")
                      .filter((col("src.PRODUCT_NAME") != col("tgt.PRODUCT_NAME")) |
                              (col("src.BASE_PRODUCT_NUMBER") != col("tgt.BASE_PRODUCT_NUMBER")) |
                              (col("src.PRODUCT_COLOR") != col("tgt.PRODUCT_COLOR")) |
                              (col("src.PRODUCT_SIZE") != col("tgt.PRODUCT_SIZE")) |
                              (col("src.PRODUCT_BRAND") != col("tgt.PRODUCT_BRAND")) |
                              (col("src.PRODUCTION_COST") != col("tgt.PRODUCTION_COST")) |
                              (col("src.INTRODUCTION_DATE") != col("tgt.INTRODUCTION_DATE")) |
                              (col("src.DISCONTINUED_DATE") != col("tgt.DISCONTINUED_DATE"))
                              ).select("src.*")
                      )

# ---------------------------------------------------------
# 4. NEW RECORDS
# ---------------------------------------------------------

new_PRODUCT_df = (df_PRODUCT_final_src.alias("src").join(df_PRODUCT_tgt.alias("tgt"),"PRODUCT_NUMBER","left_anti"))

# ---------------------------------------------------------
# 5. FINAL DELTA RECORDS
# ---------------------------------------------------------

final_PRODUCT_df = updated_PRODUCT_df.unionByName(new_PRODUCT_df)

# ---------------------------------------------------------
# 6. WRITE TO STAGING TABLE
# ---------------------------------------------------------

final_PRODUCT_df.write.jdbc(url=jdbc_url_tgt,table="STG.STG_PRODUCT",mode="overwrite",properties=connection_properties_tgt)

print("PRODUCT Dimension SCD1 staging load completed successfully.")